# SAFOD solid-Earth tides — package validation and response-model comparison

This notebook is the **presentation layer** for the Sherlock tide-model pipeline.

It does **not** install PySolid, download SPOTL, compile Fortran, or run Models A–D. Those calculations live in `scripts/tides/`.

After cloning this repository on Sherlock, regenerate everything with:

```bash
bash scripts/tides/install_spotl.sh   # first time only
bash RUN_ON_SHERLOCK.sh
```

Then reopen or re-run this notebook.

## Execution chain

$$
\text{PySolid}
\longrightarrow
\boldsymbol{\varepsilon}_{\mathrm{PySolid}}(t)
$$

$$
\text{SPOTL / ertid}
\longrightarrow
\boldsymbol{\varepsilon}_{\mathrm{SPOTL}}(t)
$$

$$
\text{transparent degree-2 model}
\longrightarrow
\boldsymbol{\varepsilon}_{\mathrm{analytic}}(t)
$$

The forcing comparison is followed by

$$
\boldsymbol{\varepsilon}
\longrightarrow
\boldsymbol{\sigma}
\longrightarrow
\text{Models A--D}
\longrightarrow
\frac{\Delta v}{v}.
$$

A package is shown as **actually run** only when its output and provenance files are present.


In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from IPython.display import display

def find_root():
    here = Path.cwd().resolve()
    for p in [here, *here.parents]:
        if (p / "config.json").exists() and (p / "scripts/tides").exists():
            return p
    raise FileNotFoundError("Could not find project root containing config.json and scripts/tides.")

ROOT = find_root()
OUT = ROOT / "outputs/tides"
CONFIG = json.loads((ROOT / "config.json").read_text())

print("Project root:", ROOT)
print("Output directory:", OUT)


## 1. Run status and provenance

This section tells us whether the plotted curves are products of the standalone Sherlock scripts.


In [ ]:
def load_json(path):
    p = Path(path)
    return json.loads(p.read_text()) if p.exists() else None

status_rows = []
for name, csv_name, prov_name in [
    ("PySolid", "pysolid_tides.csv", "pysolid_provenance.json"),
    ("SPOTL ertid", "spotl_ertid_tides.csv", "spotl_provenance.json"),
    ("analytic degree-2", "analytic_degree2_tides.csv", "analytic_degree2_provenance.json"),
    ("Models A-D", "model_results.csv", "model_provenance.json"),
]:
    csv_path = OUT / csv_name
    prov_path = OUT / prov_name
    prov = load_json(prov_path)
    status_rows.append({
        "product": name,
        "data_file_present": csv_path.exists(),
        "provenance_present": prov_path.exists(),
        "hostname": None if prov is None else prov.get("hostname"),
        "created_utc": None if prov is None else prov.get("created_utc"),
        "git_commit": None if prov is None else prov.get("git_commit"),
    })
status = pd.DataFrame(status_rows)
display(status)


## 2. Package forcing

### PySolid

PySolid is an IERS-style solid-Earth-tide implementation. The standalone script evaluates ENU displacement at SAFOD and at a small spatial stencil; those spatial derivatives provide the horizontal strain tensor.

### SPOTL / `ertid`

SPOTL `ertid` computes body tides directly from Sun/Moon positions. The script requests extensional strain at $0^\circ$, $45^\circ$, and $90^\circ$ and reconstructs

$$
\varepsilon_{NN},
\qquad
\varepsilon_{EE},
\qquad
\varepsilon_{NE}.
$$

If the SPOTL output file is absent below, that means the standalone `ertid` script has not yet been run in this copy of the framework. No surrogate is silently substituted.


In [ ]:
def read_product(name):
    path = OUT / name
    if not path.exists():
        return None
    return pd.read_csv(path, parse_dates=["time_utc"])

pysolid = read_product("pysolid_tides.csv")
spotl = read_product("spotl_ertid_tides.csv")
analytic = read_product("analytic_degree2_tides.csv")

print("PySolid:", "NOT PRESENT — run pipeline" if pysolid is None else f"{len(pysolid)} samples")
print("SPOTL:", "NOT PRESENT — run pipeline" if spotl is None else f"{len(spotl)} samples")
print("analytic:", "NOT PRESENT — run pipeline" if analytic is None else f"{len(analytic)} samples")


In [ ]:
if pysolid is None:
    print("No forcing products yet. Run: bash RUN_ON_SHERLOCK.sh")
else:
    plt.figure(figsize=(11,5))
    plt.plot(pysolid.time_utc, 1e9*(pysolid.areal_strain-pysolid.areal_strain.mean()), linewidth=2, label="PySolid")
    if spotl is not None:
        plt.plot(spotl.time_utc, 1e9*(spotl.areal_strain-spotl.areal_strain.mean()), label="SPOTL ertid")
    if analytic is not None and "areal_strain" in analytic:
        plt.plot(analytic.time_utc, 1e9*(analytic.areal_strain-analytic.areal_strain.mean()), label="transparent degree-2")
    plt.axhline(0,linewidth=0.8)
    plt.ylabel("Mean-removed areal strain (nanostrain)")
    plt.xlabel("Time (UTC)")
    plt.title("Independent solid-Earth-tide forcing calculations")
    plt.legend(); plt.grid(alpha=0.25)
    plt.gca().xaxis.set_major_locator(mdates.HourLocator(interval=3))
    plt.gca().xaxis.set_major_formatter(mdates.DateFormatter("%m-%d %H:%M"))
    plt.xticks(rotation=30,ha="right"); plt.tight_layout(); plt.show()


In [ ]:
comparison_path = OUT / "forcing_comparison.csv"
if comparison_path.exists():
    comparison = pd.read_csv(comparison_path)
    display(comparison)
else:
    print("forcing_comparison.csv is not present yet. It is produced after both PySolid and SPOTL have been run.")


## 3. Response models

The standalone model script evaluates the same four branches for each available package forcing.

### Model A — Niu amplitude shortcut

$$
\Delta\sigma_A(t)=240\,\mathrm{Pa}\,\tau(t),
$$

$$
\left(\frac{\Delta v}{v}\right)_A=S_{\mathrm{Niu}}\Delta\sigma_A.
$$

### Model B — strain to elastic fault stress

$$
\boldsymbol{\varepsilon}\longrightarrow\boldsymbol{\sigma}\longrightarrow\Delta\sigma_n\longrightarrow S_{\mathrm{Niu}}\longrightarrow\frac{\Delta v}{v}.
$$

The current constitutive closure is a **surface plane-stress scenario**, not a rigorous $1\,\mathrm{km}$ stress tensor.

### Model C — direct published strain sensitivity

This applies the Takano / Sheng style strain coefficient directly and is kept as a literature-transfer comparison rather than expected SAFOD behavior.

### Model D — crack-closure effective-medium model

$$
\Delta\sigma\longrightarrow\rho_c(\sigma)\longrightarrow K_{\mathrm{eff}},\mu_{\mathrm{eff}}\longrightarrow V_P,V_S.
$$

The operating crack density is constrained with the site-informed $V_S$; the crack-closure stress scale is calibrated to the local Niu stress sensitivity. It is a mechanistic formation-rock model, not yet an AWD guided-mode model.


In [ ]:
model_path = OUT / "model_results.csv"
models = pd.read_csv(model_path, parse_dates=["time_utc"]) if model_path.exists() else None
summary = pd.DataFrame()

if models is None:
    print("No model products yet. Run: bash RUN_ON_SHERLOCK.sh")
else:
    summary_rows = []
    for forcing in ["pysolid","spotl"]:
        mapping = {
            "A": f"{forcing}_model_A_dv_over_v",
            "B": f"{forcing}_model_B_dv_over_v",
            "C": f"{forcing}_model_C_takano_dv_over_v",
            "D_Vs": f"{forcing}_model_D_dVs_over_Vs",
            "D_Vp": f"{forcing}_model_D_dVp_over_Vp",
        }
        if all(col in models.columns for col in mapping.values()):
            for model_name,col in mapping.items():
                summary_rows.append({
                    "forcing": forcing, "model": model_name,
                    "max_abs_fraction": np.nanmax(np.abs(models[col])),
                    "max_abs_percent": 100*np.nanmax(np.abs(models[col])),
                })
    summary = pd.DataFrame(summary_rows)
    display(summary.style.format({"max_abs_fraction":"{:.3e}","max_abs_percent":"{:.5f}"}))


In [ ]:
if models is None:
    print("No model products yet. Run: bash RUN_ON_SHERLOCK.sh")
else:
    plt.figure(figsize=(11,5))
    for forcing,ls in [("pysolid","-"),("spotl","--")]:
        b=f"{forcing}_model_B_dv_over_v"
        d=f"{forcing}_model_D_dVs_over_Vs"
        if b in models: plt.plot(models.time_utc,100*models[b],ls,label=f"Model B — {forcing}")
        if d in models: plt.plot(models.time_utc,100*models[d],ls,label=f"Model D Vs — {forcing}")
    plt.axhline(0,linewidth=0.8); plt.ylabel(r"$\Delta v/v$ (%)"); plt.xlabel("Time (UTC)")
    plt.title("Mechanically explicit response models"); plt.legend(); plt.grid(alpha=0.25)
    plt.gca().xaxis.set_major_locator(mdates.HourLocator(interval=3))
    plt.gca().xaxis.set_major_formatter(mdates.DateFormatter("%m-%d %H:%M"))
    plt.xticks(rotation=30,ha="right"); plt.tight_layout(); plt.show()


## 4. AWD detectability comparison

The strongest empirical AWD benchmark is the Deep outbound reliable tested change,

$$
\left|\frac{\Delta v}{v}\right|=5\times10^{-3}=0.5\%.
$$

This is compared with the model amplitudes only as an order-of-magnitude detectability benchmark. It is not a tide detection and is not a theoretical noise floor.


In [ ]:
if summary.empty:
    print("No model summary yet. Run: bash RUN_ON_SHERLOCK.sh")
else:
    awd = CONFIG["awd_benchmarks"]
    detect = summary.copy()
    detect["deep_outbound_reliable_over_model"] = awd["deep_outbound_reliable"] / detect["max_abs_fraction"]
    display(detect.style.format({
        "max_abs_fraction":"{:.3e}",
        "max_abs_percent":"{:.5f}",
        "deep_outbound_reliable_over_model":"{:.1f}x",
    }))


## 5. Interpretation

The intended scientific hierarchy is:

1. **Package disagreement** quantifies uncertainty in the body-tide forcing.
2. **Model B** is the main mechanical stress scenario.
3. **Model A** is the simple SAFOD empirical benchmark.
4. **Model C** is an intentionally aggressive foreign-site strain-transfer comparison.
5. **Model D** asks whether an explicit crack-closure mechanism can reproduce the observed scale and what rock-physics parameters are required.

The largest remaining uncertainty is expected to be the **local constitutive response and stress boundary condition**, not the astronomical timing of the body tide.

### Claims intentionally not made

- No tidal response is detected in the AWD data.
- The Niu $240\,\mathrm{Pa}$ number is not assigned an undocumented stress component.
- The plane-stress Model B stress is not called the exact stress tensor at $1\,\mathrm{km}$.
- Model D formation $V_P/V_S$ is not equated with the AWD guided-mode apparent velocity.


## 6. Key references

- Agnew, D. C. (2012). *SPOTL: Some Programs for Ocean-Tide Loading*. Scripps Institution of Oceanography.
- Métivier, L., & Conrad, C. P. (2008). DOI: `10.1029/2007JB005448`.
- Thomas, A. M., et al. (2012). DOI: `10.1029/2011JB009036`.
- Johnson, C. W., Fu, Y., & Bürgmann, R. (2017). DOI: `10.1002/2017JB014778`.
- Boness, N. L., & Zoback, M. D. (2004). DOI: `10.1029/2003GL019020`.
- Niu, F., et al. (2008). DOI: `10.1038/nature07111`.
- Silver, P. G., et al. (2007). DOI: `10.1785/0120060120`.
- Takano, T., et al. (2014). DOI: `10.1002/2014GL060690`.
- Ba, J., et al. (2024). DOI: `10.1093/gji/ggae020`.
